# 06 - Embedding Experiments

### Objective
To explore different word embedding techniques for representing
multilingual civic complaints and identify suitable embeddings
for the LSTM, BiLSTM and Attention-BiLSTM models.

In [1]:
# Import libraries for loading the saved preprocessing artifacts
import os
import pickle
import numpy as np
import pandas as pd

# Define the directory containing the saved preprocessing files
PREPROCESSING_DIR = "../models/preprocessing"

# Load the fitted tokenizer
with open(os.path.join(PREPROCESSING_DIR, "tokenizer.pkl"), "rb") as file:
    tokenizer = pickle.load(file)

# Load the fitted label encoder
with open(os.path.join(PREPROCESSING_DIR, "label_encoder.pkl"), "rb") as file:
    label_encoder = pickle.load(file)

# Load the maximum sequence length
MAX_LEN = int(np.load(os.path.join(PREPROCESSING_DIR, "max_len.npy")))

# Load the tokenized training, validation and test datasets
tokenized_data = np.load(
    os.path.join(PREPROCESSING_DIR, "tokenized_data.npz")
)

# Extract the prepared arrays
X_train = tokenized_data["X_train"]
X_val = tokenized_data["X_val"]
X_test = tokenized_data["X_test"]

y_train = tokenized_data["y_train"]
y_val = tokenized_data["y_val"]
y_test = tokenized_data["y_test"]

# Display the loaded dataset dimensions and vocabulary information
print("Vocabulary Size:", len(tokenizer.word_index) + 1)
print("Maximum Sequence Length:", MAX_LEN)
print("Number of Classes:", len(label_encoder.classes_))

print("\nTraining Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)
print("Testing Shape:", X_test.shape)

Vocabulary Size: 20089
Maximum Sequence Length: 156
Number of Classes: 12

Training Shape: (11563, 156)
Validation Shape: (2478, 156)
Testing Shape: (2478, 156)


In [3]:
import gensim

## 2. Training Word2Vec Embeddings

In [4]:
# Import the Word2Vec implementation from Gensim
from gensim.models import Word2Vec

# Convert padded training sequences back into word tokens
# Padding index 0 is excluded, and unknown index 1 is retained as <OOV>
index_word = {index: word for word, index in tokenizer.word_index.items()}

# Create tokenized sentences using the original training sequences
train_sentences = [
    [index_word.get(token, "<OOV>") for token in sequence if token != 0]
    for sequence in X_train
]

# Display a sample tokenized complaint
print("Sample Tokenized Complaint:")
print(train_sentences[0])

# Train Word2Vec using the training complaints
word2vec_model = Word2Vec(
    sentences=train_sentences,  # Training corpus
    vector_size=100,             # Dimension of each word vector
    window=5,                    # Context window size
    min_count=2,                 # Ignore words occurring fewer than twice
    workers=4,                   # Number of CPU workers
    sg=1,                        # Use Skip-gram architecture
    epochs=10,                   # Number of training iterations
    seed=42                      # Reproducibility
)

# Display Word2Vec vocabulary information
print("\nWord2Vec Vocabulary Size:", len(word2vec_model.wv))
print("Embedding Dimension:", word2vec_model.vector_size)

Sample Tokenized Complaint:
['parking', 'under', 'no', 'parking', 'on', 'footpath', 'blocking', 'the', 'entry', 'of', 'resi', 'parking', 'under', 'no', 'parking', 'on', 'footpath', 'blocking', 'the', 'entry', 'of', 'residence', 'on', 'foot', 'path', 'violation', 'of', 'traffic', 'regulations', 'and', 'public', 'nuisance']

Word2Vec Vocabulary Size: 11087
Embedding Dimension: 100


In [5]:
# Define words related to common civic complaints
test_words = [
    "road",
    "garbage",
    "drainage",
    "water",
    "street",
    "traffic",
    "electricity",
    "footpath"
]

# Display similar words learned by the Word2Vec model
for word in test_words:

    # Check whether the word exists in the Word2Vec vocabulary
    if word in word2vec_model.wv:

        # Retrieve the five most similar words
        similar_words = word2vec_model.wv.most_similar(word, topn=5)

        print(f"\nWords similar to '{word}':")

        # Display each similar word and its similarity score
        for similar_word, score in similar_words:
            print(f"{similar_word}: {score:.4f}")

    else:
        # Report words that were not learned by Word2Vec
        print(f"\n'{word}' is not present in the Word2Vec vocabulary.")


Words similar to 'road':
strech: 0.7213
roads: 0.7198
streach: 0.7188
jail: 0.7131
wh: 0.7061

Words similar to 'garbage':
trash: 0.8063
garbages: 0.7638
garabage: 0.7193
grabage: 0.7188
gabage: 0.7175

Words similar to 'drainage':
sewage: 0.7638
drinage: 0.7334
overflow: 0.7284
drain: 0.7234
outflow: 0.7223

Words similar to 'water':
sump: 0.7104
leaking: 0.6934
septic: 0.6824
harvesting: 0.6821
flowing: 0.6764

Words similar to 'street':
tube: 0.6920
functioning: 0.6859
led: 0.6806
halogen: 0.6794
2months: 0.6751

Words similar to 'traffic':
jam: 0.8334
jams: 0.7447
congestion: 0.7173
chaos: 0.6822
neck: 0.6577

Words similar to 'electricity':
power: 0.7924
shutdown: 0.7274
8hrs: 0.7090
disconnected: 0.7050
outage: 0.7020

Words similar to 'footpath':
footpaths: 0.6920
sidewalk: 0.6783
bollard: 0.6313
placed: 0.6308
overgrowth: 0.6249


In [6]:
# Define the embedding dimension used by the Word2Vec model
EMBEDDING_DIM = 100

# Get the tokenizer vocabulary size, including the padding index
VOCAB_SIZE = len(tokenizer.word_index) + 1

# Initialize an embedding matrix with zeros
# Index 0 is reserved for padding
embedding_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM))

# Count words that have matching Word2Vec embeddings
matched_words = 0

# Map each tokenizer word to its corresponding Word2Vec vector
for word, index in tokenizer.word_index.items():

    # Check whether the word exists in the Word2Vec vocabulary
    if word in word2vec_model.wv:

        # Store the pretrained vector at the tokenizer's word index
        embedding_matrix[index] = word2vec_model.wv[word]

        # Increment the matched word counter
        matched_words += 1

# Calculate embedding coverage
coverage = (matched_words / len(tokenizer.word_index)) * 100

# Display embedding matrix information
print("Embedding Matrix Shape:", embedding_matrix.shape)
print("Matched Words:", matched_words)
print("Tokenizer Vocabulary:", len(tokenizer.word_index))
print("Embedding Coverage: {:.2f}%".format(coverage))

Embedding Matrix Shape: (20089, 100)
Matched Words: 11087
Tokenizer Vocabulary: 20088
Embedding Coverage: 55.19%


In [7]:
# Create a directory for storing Word2Vec artifacts
os.makedirs("../models/embeddings", exist_ok=True)

# Save the trained Word2Vec model
word2vec_model.save("../models/embeddings/civicsense_word2vec.model")

# Save the embedding matrix as a NumPy file
np.save(
    "../models/embeddings/word2vec_embedding_matrix.npy",
    embedding_matrix
)

# Save embedding configuration and coverage information
embedding_info = {
    "embedding_dimension": EMBEDDING_DIM,
    "vocabulary_size": VOCAB_SIZE,
    "matched_words": matched_words,
    "embedding_coverage": coverage,
    "max_sequence_length": MAX_LEN
}

# Store the configuration for future experiments
with open("../models/embeddings/word2vec_info.pkl", "wb") as file:
    pickle.dump(embedding_info, file)

# Confirm successful saving
print("Word2Vec model and embedding artifacts saved successfully.")

Word2Vec model and embedding artifacts saved successfully.


## 3. Embedding Strategy Comparison.

In [8]:
# Import TensorFlow and evaluation utilities
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score, f1_score

# Set random seeds for reproducible experiments
np.random.seed(42)
tf.random.set_seed(42)

# Load the saved tokenized datasets
PREPROCESSING_DIR = "../models/preprocessing"

tokenized_data = np.load(
    os.path.join(PREPROCESSING_DIR, "tokenized_data.npz")
)

# Extract training and validation datasets
X_train = tokenized_data["X_train"]
X_val = tokenized_data["X_val"]

y_train = tokenized_data["y_train"]
y_val = tokenized_data["y_val"]

# Load the saved Word2Vec embedding matrix
embedding_matrix = np.load(
    "../models/embeddings/word2vec_embedding_matrix.npy"
)

# Define common model configuration
VOCAB_SIZE = embedding_matrix.shape[0]
EMBEDDING_DIM = embedding_matrix.shape[1]
MAX_LEN = X_train.shape[1]
NUM_CLASSES = len(label_encoder.classes_)

# Display the experiment configuration
print("Training Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)
print("Word2Vec Matrix Shape:", embedding_matrix.shape)
print("Embedding Dimension:", EMBEDDING_DIM)
print("Number of Classes:", NUM_CLASSES)

Training Shape: (11563, 156)
Validation Shape: (2478, 156)
Word2Vec Matrix Shape: (20089, 100)
Embedding Dimension: 100
Number of Classes: 12


In [9]:
# Define a reusable function to build the LSTM model
def build_lstm_model(embedding_type="random"):

    # Initialize the embedding layer with random weights
    if embedding_type == "random":
        embedding_layer = Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBEDDING_DIM,
            embeddings_initializer=tf.keras.initializers.RandomUniform(seed=42),
            mask_zero=True,
            trainable=True
        )

    # Initialize the embedding layer using pretrained Word2Vec vectors
    elif embedding_type == "word2vec":
        embedding_layer = Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBEDDING_DIM,
            weights=[embedding_matrix],
            mask_zero=True,
            trainable=True
        )

    else:
        # Reject unsupported embedding strategies
        raise ValueError("Choose either 'random' or 'word2vec'.")

    # Create the LSTM classification model
    model = Sequential([
        embedding_layer,
        LSTM(64),
        Dropout(0.3),
        Dense(NUM_CLASSES, activation="softmax")
    ])

    # Configure the model for multiclass classification
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# Build both models using the same architecture
random_model = build_lstm_model("random")
word2vec_model_lstm = build_lstm_model("word2vec")

# Display the architecture of the random embedding model
random_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Explicitly build both models using the defined input sequence length
random_model.build(input_shape=(None, MAX_LEN))
word2vec_model_lstm.build(input_shape=(None, MAX_LEN))

# Display the random embedding model architecture
random_model.summary()

# Display the Word2Vec embedding model architecture
word2vec_model_lstm.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 156, 100)       │     2,008,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        42,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 12)             │           780 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,051,920 (7.83 MB)

 Trainable params: 2,051,920 (7.83 MB)

 Non-trainable params: 0 (0.00 B)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 156, 100)       │     2,008,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        42,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 12)             │           780 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,051,920 (7.83 MB)

 Trainable params: 2,051,920 (7.83 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Define the early stopping callback to prevent unnecessary training
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Train the model with randomly initialized embeddings
random_history = random_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

# Display the final training and validation accuracy
print("\nRandom Embedding Model Results")
print("Training Accuracy:", random_history.history["accuracy"][-1])
print("Validation Accuracy:", random_history.history["val_accuracy"][-1])

Epoch 1/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 23s 113ms/step - accuracy: 0.5109 - loss: 1.6143 - val_accuracy: 0.6538 - val_loss: 1.1681
Epoch 2/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 20s 108ms/step - accuracy: 0.7089 - loss: 0.9955 - val_accuracy: 0.7127 - val_loss: 1.0037
Epoch 3/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 19s 107ms/step - accuracy: 0.7965 - loss: 0.7419 - val_accuracy: 0.7583 - val_loss: 1.0347
Epoch 4/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 20s 109ms/step - accuracy: 0.8542 - loss: 0.5769 - val_accuracy: 0.7893 - val_loss: 0.8502
Epoch 5/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 19s 105ms/step - accuracy: 0.8562 - loss: 0.5387 - val_accuracy: 0.7672 - val_loss: 0.8594
Epoch 6/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 20s 111ms/step - accuracy: 0.9023 - loss: 0.3679 - val_accuracy: 0.7841 - val_loss: 0.8432
Epoch 7/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 20s 106ms/step - accuracy: 0.9208 - loss: 0.2870 - val_accuracy: 0.7857 - val_loss: 0.9008
Epoch 8/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 20s 110ms/step - accuracy: 0.9295 - loss: 0

In [12]:
# Evaluate the random embedding model using its restored best weights
random_train_loss, random_train_acc = random_model.evaluate(
    X_train, y_train, verbose=0
)

random_val_loss, random_val_acc = random_model.evaluate(
    X_val, y_val, verbose=0
)

# Calculate validation predictions and macro F1-score
random_val_predictions = np.argmax(
    random_model.predict(X_val, verbose=0), axis=1
)

random_val_f1 = f1_score(
    y_val, random_val_predictions, average="macro"
)

# Display the final evaluation metrics
print("Random Embedding Model - Restored Best Weights")
print(f"Training Accuracy: {random_train_acc:.4f}")
print(f"Validation Accuracy: {random_val_acc:.4f}")
print(f"Validation Macro F1-score: {random_val_f1:.4f}")

Random Embedding Model - Restored Best Weights
Training Accuracy: 0.9242
Validation Accuracy: 0.7841
Validation Macro F1-score: 0.6150


In [13]:
# Create a separate early stopping callback for the Word2Vec experiment
word2vec_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Train the Word2Vec-initialized LSTM using identical settings
word2vec_history = word2vec_model_lstm.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[word2vec_early_stopping],
    verbose=1
)

# Evaluate the restored best model on training data
w2v_train_loss, w2v_train_acc = word2vec_model_lstm.evaluate(
    X_train, y_train, verbose=0
)

# Evaluate the restored best model on validation data
w2v_val_loss, w2v_val_acc = word2vec_model_lstm.evaluate(
    X_val, y_val, verbose=0
)

# Calculate validation predictions
w2v_val_predictions = np.argmax(
    word2vec_model_lstm.predict(X_val, verbose=0), axis=1
)

# Calculate macro F1-score across all 12 categories
w2v_val_f1 = f1_score(
    y_val, w2v_val_predictions, average="macro"
)

# Display the Word2Vec evaluation metrics
print("\nWord2Vec Embedding Model - Restored Best Weights")
print(f"Training Accuracy: {w2v_train_acc:.4f}")
print(f"Validation Accuracy: {w2v_val_acc:.4f}")
print(f"Validation Macro F1-score: {w2v_val_f1:.4f}")

Epoch 1/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 22s 112ms/step - accuracy: 0.5872 - loss: 1.4000 - val_accuracy: 0.7119 - val_loss: 0.9699
Epoch 2/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 20s 108ms/step - accuracy: 0.7558 - loss: 0.8739 - val_accuracy: 0.7462 - val_loss: 0.8798
Epoch 3/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 22s 122ms/step - accuracy: 0.8097 - loss: 0.7083 - val_accuracy: 0.7889 - val_loss: 0.7389
Epoch 4/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 23s 125ms/step - accuracy: 0.8423 - loss: 0.5872 - val_accuracy: 0.8128 - val_loss: 0.6892
Epoch 5/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 20s 110ms/step - accuracy: 0.8681 - loss: 0.4898 - val_accuracy: 0.8063 - val_loss: 0.7003
Epoch 6/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 19s 106ms/step - accuracy: 0.8813 - loss: 0.4390 - val_accuracy: 0.8140 - val_loss: 0.6705
Epoch 7/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 19s 103ms/step - accuracy: 0.8453 - loss: 0.5565 - val_accuracy: 0.8010 - val_loss: 0.7228
Epoch 8/15
181/181 ━━━━━━━━━━━━━━━━━━━━ 19s 107ms/step - accuracy: 0.8923 - loss: 0

In [14]:
# Create a directory for storing embedding comparison results
RESULTS_DIR = "../models/experiments/embedding_comparison"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Create a comparison table using restored best-model evaluation metrics
embedding_results = pd.DataFrame({
    "Embedding Strategy": ["Random", "Word2Vec"],
    "Training Accuracy": [random_train_acc, w2v_train_acc],
    "Validation Accuracy": [random_val_acc, w2v_val_acc],
    "Validation Macro F1": [random_val_f1, w2v_val_f1]
})

# Display the comparison table
display(embedding_results.round(4))

# Save the comparison results as a CSV file
embedding_results.to_csv(
    os.path.join(RESULTS_DIR, "embedding_comparison.csv"),
    index=False
)

# Save both trained models for future reference
random_model.save(os.path.join(RESULTS_DIR, "random_embedding_lstm.keras"))
word2vec_model_lstm.save(os.path.join(RESULTS_DIR, "word2vec_embedding_lstm.keras"))

# Save training histories for later analysis
with open(os.path.join(RESULTS_DIR, "training_histories.pkl"), "wb") as file:
    pickle.dump({
        "random": random_history.history,
        "word2vec": word2vec_history.history
    }, file)

# Confirm that the comparison artifacts have been saved
print("Embedding comparison results and models saved successfully.")


,Embedding Strategy,Training Accuracy,Validation Accuracy,Validation Macro F1
0,Random,0.9242,0.7841,0.6150
1,Word2Vec,0.9031,0.8140,0.6659


Embedding comparison results and models saved successfully.
